In [ ]:
import os
import pickle
import sys
import time

root_dir = os.path.abspath("..")
data_dir = os.path.join(root_dir, "data")
os.makedirs(data_dir, exist_ok=True)
if root_dir not in sys.path:
    sys.path.append(root_dir)

from current_setpoints.models.machines import ieee_machine2_trained_neural_flux
from current_setpoints.models.forward_model import ForwardModel, Fault
from current_setpoints.optimization import (
    StaticOptimizer,
    ActiveSetOptimizer,
    FeasibilityOptimizer,
    calculate_grid,
    grid_to_data,
)
from current_setpoints.utils.plotting import plot_grid_segments

In [ ]:
CURR_MAX = 30.0
VOLT_MAX = 13.0
OMEGA_MAX = 1800.0

SLSQP_OPTS = {"disp": False, "ftol": 1e-8, "maxiter": 500}
N_GRID = 60

GRID_OPTS = {"n_torq": 101, "n_omega": 101, "torq_min": 0.0, "omega_min": 0.0, "omega_max": OMEGA_MAX}

drive = ieee_machine2_trained_neural_flux(curr_max=CURR_MAX, volt_max=VOLT_MAX)
fwd = ForwardModel(drive, n_theta=700, fault=Fault(()))

print(f"healthy mode, neural flux, {GRID_OPTS['n_torq']}x{GRID_OPTS['n_omega']} grid, N_GRID={N_GRID}")

In [ ]:
static_pkl = os.path.join(data_dir, "torque_speed_map_static_neural.pkl")

if os.path.isfile(static_pkl):
    with open(static_pkl, "rb") as f:
        grid_static = pickle.load(f)
    print(f"loaded cached static map from {static_pkl}")
else:
    optimizer_static = StaticOptimizer(fwd, opts=SLSQP_OPTS)
    t0 = time.perf_counter()
    grid_static = calculate_grid(optimizer=optimizer_static, fwd=fwd, opts=GRID_OPTS, mode="standard")
    print(f"static map done in {time.perf_counter() - t0:.0f}s")
    with open(static_pkl, "wb") as f:
        pickle.dump(grid_static, f)
    print(f"saved to {static_pkl}")

In [ ]:
dynamic_old_pkl = os.path.join(data_dir, "torque_speed_map_dynamic_old_neural.pkl")

if os.path.isfile(dynamic_old_pkl):
    with open(dynamic_old_pkl, "rb") as f:
        grid_dynamic_old = pickle.load(f)
    print(f"loaded cached dynamic-old map from {dynamic_old_pkl}")
else:
    optimizer_dynamic_old = ActiveSetOptimizer(fwd, opts=SLSQP_OPTS, n_grid=N_GRID)
    t0 = time.perf_counter()
    grid_dynamic_old = calculate_grid(optimizer=optimizer_dynamic_old, fwd=fwd, opts=GRID_OPTS, mode="standard")
    print(f"dynamic-old (ActiveSetOptimizer) map done in {time.perf_counter() - t0:.0f}s")
    with open(dynamic_old_pkl, "wb") as f:
        pickle.dump(grid_dynamic_old, f)
    print(f"saved to {dynamic_old_pkl}")

In [ ]:
dynamic_new_pkl = os.path.join(data_dir, "torque_speed_map_dynamic_new_neural.pkl")

if os.path.isfile(dynamic_new_pkl):
    with open(dynamic_new_pkl, "rb") as f:
        grid_dynamic_new = pickle.load(f)
    print(f"loaded cached dynamic-new map from {dynamic_new_pkl}")
else:
    optimizer_dynamic_new = FeasibilityOptimizer(fwd, opts=SLSQP_OPTS, n_grid=N_GRID)
    t0 = time.perf_counter()
    grid_dynamic_new = calculate_grid(optimizer=optimizer_dynamic_new, fwd=fwd, opts=GRID_OPTS, mode="standard")
    print(f"dynamic-new (FeasibilityOptimizer) map done in {time.perf_counter() - t0:.0f}s")
    with open(dynamic_new_pkl, "wb") as f:
        pickle.dump(grid_dynamic_new, f)
    print(f"saved to {dynamic_new_pkl}")

In [ ]:
plot_grid_segments(grid_to_data(grid_static, k_skip=1), title="Static mode (neural flux, healthy)")
plot_grid_segments(grid_to_data(grid_dynamic_old, k_skip=1), title="Dynamic mode, old optimizer (ActiveSetOptimizer, neural flux, healthy)")
plot_grid_segments(grid_to_data(grid_dynamic_new, k_skip=1), title="Dynamic mode, new optimizer (FeasibilityOptimizer, neural flux, healthy)")